# 1.Data Preprocessing

In [ ]:
from numpy import astype
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os


file_path = "all-data.csv"
df = pd.read_csv("/content/all-data.csv", header=None, encoding="latin1")
df.columns = ["label", "text"]

print("the first 5 row of the original data:")
print(df.head())
print("\n original data shape:", df.shape)

#clearing
print("\n ---checking---")
print("null values:")
print(df.isnull().sum())

print("\n number of duplicate row:", df.duplicated().sum())


df = df.drop_duplicates().reset_index(drop=True)
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.lower()

df = df[df["text"] != ""].reset_index(drop=True)


print("\n ----result-----")
print("Data shape after cleaning:", df.shape)
print("label category", df["label"].unique())

valid_labels = ["positive", "negative", "neutral"]
df = df[df["label"].isin(valid_labels)].reset_index(drop=True)

print("\n final data shape：", df.shape)

### Statistical category distribution


In [ ]:
label_counts = df["label"].value_counts()
label_percent = df["label"].value_counts(normalize=True) * 100

distribution_report = pd.DataFrame({
    "count": label_counts,
    "percentage": label_percent.round(2)
})

print("\n=====Category distribution statistics=====")
print(distribution_report)

total_samples = len(df)
print(f"\nTotal number of samples: {total_samples}")


output_dir = "processed_data"
os.makedirs(output_dir, exist_ok=True)

cleaned_file = os.path.join(output_dir, "cleaned_all_data.csv")
df.to_csv(cleaned_file, index=False, encoding="utf-8-sig")
print(f"\n cleaning values has been saved：{cleaned_file}")

# 5. Visualize category distribution (bar chart)
plt.figure(figsize=(8, 5))
label_counts.plot(kind="bar")
plt.title("Sentiment Class Distribution")
plt.xlabel("Sentiment Label")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()

plot_file = os.path.join(output_dir, "class_distribution_bar.png")
plt.savefig(plot_file, dpi=300)
plt.show()
print(f"The category distribution bar chart has been saved：{plot_file}")


### Slipt data 7:2:1 trian : validation : test

In [ ]:
random_seed = 42

train_val_df, test_df = train_test_split(
    df,
    test_size=0.1,
    random_state=random_seed,
    stratify=df["label"]
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=2/9,   # Aim for a final ratio close to 7:2:1
    random_state=random_seed,
    stratify=train_val_df["label"]
)

print("\n=====Data set partition results =====")
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

print("\nTrain label distribution:")
print(train_df["label"].value_counts(normalize=True).round(4))

print("\nValidation label distribution:")
print(val_df["label"].value_counts(normalize=True).round(4))

print("\nTest label distribution:")
print(test_df["label"].value_counts(normalize=True).round(4))

# -------------------------
# export CSV
train_file = os.path.join(output_dir, "train.csv")
val_file = os.path.join(output_dir, "validation.csv")
test_file = os.path.join(output_dir, "test.csv")

train_df.to_csv(train_file, index=False, encoding="utf-8-sig")
val_df.to_csv(val_file, index=False, encoding="utf-8-sig")
test_df.to_csv(test_file, index=False, encoding="utf-8-sig")

print(f"\ntrain.csv：{train_file}")
print(f"validation.csv：{val_file}")
print(f"test.csv：{test_file}")


### Data Distribution Report

In [ ]:
report_file = os.path.join(output_dir, "data_distribution_report.txt")

with open(report_file, "w", encoding="utf-8") as f:
    f.write("Sentiment Analysis for Financial News - Data Report\n")
    f.write("=================================================\n\n")
    f.write(f"Total samples after cleaning: {len(df)}\n\n")

    f.write("Class distribution:\n")
    for label in distribution_report.index:
        count = distribution_report.loc[label, "count"]
        percent = distribution_report.loc[label, "percentage"]
        f.write(f"- {label}: {count} samples ({percent}%)\n")

    f.write("\nObservation:\n")
    f.write("The dataset is imbalanced. The neutral class has the largest number of samples, ")
    f.write("while the negative class has the fewest. This may affect model training and is one ")
    f.write("reason for trying methods such as Focal Loss in the next stage.\n\n")

    f.write("Data split ratio:\n")
    f.write(f"- Train: {len(train_df)}\n")
    f.write(f"- Validation: {len(val_df)}\n")
    f.write(f"- Test: {len(test_df)}\n")
    f.write("\nRandom seed: 42\n")

print(f"The text report has been saved：{report_file}")


### Convert to Hugging Face Dataset Format

In [ ]:
!pip install datasets
from datasets import Dataset, DatasetDict

label2id = {"negative": 0, "neutral": 1, "positive": 2}

train_hf = train_df.copy()
val_hf = val_df.copy()
test_hf = test_df.copy()

train_hf["label"] = train_hf["label"].map(label2id)
val_hf["label"] = val_hf["label"].map(label2id)
test_hf["label"] = test_hf["label"].map(label2id)

# save text and label
train_hf = train_hf[["text", "label"]]
val_hf = val_hf[["text", "label"]]
test_hf = test_hf[["text", "label"]]

print("train_hf columns:", train_hf.columns.tolist())
print("val_hf columns:", val_hf.columns.tolist())
print("test_hf columns:", test_hf.columns.tolist())

#Hugging Face Dataset
dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_hf, preserve_index=False),
    "validation": Dataset.from_pandas(val_hf, preserve_index=False),
    "test": Dataset.from_pandas(test_hf, preserve_index=False)
})

print("\nHugging Face Dataset")
print(dataset_dict)

hf_dir = os.path.join(output_dir, "hf_dataset")
dataset_dict.save_to_disk(hf_dir)
print(f"Hugging Face Dataset：{hf_dir}")

# 2.Model disign and Training




### Set up the FinBert Library

In [ ]:
from transformers import AutoTokenizer

# Load the pre-trained FinBERT tokenizer
tokenizer_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# Define the tokenization function
def tokenize_function(examples):
    # padding="max_length" ensures all sequences have the same length
    # truncation=True cuts off sequences that exceed the maximum length
    # A max_length of 128 is generally sufficient for financial news headlines
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Apply the tokenization function to the dataset in batches
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

# Remove the original text column and set the output format to PyTorch tensors
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")

print(tokenized_datasets)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=BATCH_SIZE)
val_dataloader = DataLoader(tokenized_datasets["validation"], batch_size=BATCH_SIZE)
test_dataloader = DataLoader(tokenized_datasets["test"], batch_size=BATCH_SIZE)

print(f"Number of training batches: {len(train_dataloader)}")

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

class FinBert_BiLSTM_Attention(nn.Module):
    def __init__(self, model_name="ProsusAI/finbert", hidden_dim=256, num_classes=3):
        super(FinBert_BiLSTM_Attention, self).__init__()

        # 1. FinBERT Layer (Responsible for extracting deep semantic features)
        self.bert = AutoModel.from_pretrained(model_name)

        # 2. BiLSTM Layer (Captures contextual and sequential relationships)
        self.lstm = nn.LSTM(input_size=self.bert.config.hidden_size,
                            hidden_size=hidden_dim,
                            num_layers=1,
                            batch_first=True,
                            bidirectional=True)

        # 3. Self-Attention Layer
        # Calculates the weight of each word, focusing on financial sentiment keywords
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

        # 4. Fully Connected Classification Layer (Outputs 3 classes: negative, neutral, positive)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

        # Dropout layer to prevent overfitting
        self.dropout = nn.Dropout(0.3)

    def forward(self, input_ids, attention_mask):
        # Obtain FinBERT outputs
        # sequence_output shape: [batch_size, seq_len, 768]
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_outputs.last_hidden_state

        # Pass through BiLSTM
        # lstm_out shape: [batch_size, seq_len, hidden_dim * 2]
        lstm_out, _ = self.lstm(sequence_output)

        # Calculate Attention weights
        # attn_weights shape: [batch_size, seq_len, 1]
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)

        # Multiply weights with BiLSTM outputs and sum to get the final sentence representation (Context Vector)
        # context_vector shape: [batch_size, hidden_dim * 2]
        context_vector = torch.sum(attn_weights * lstm_out, dim=1)

        # Apply Dropout and then pass through the classifier
        context_vector = self.dropout(context_vector)
        logits = self.classifier(context_vector)

        return logits

# Instantiate the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FinBert_BiLSTM_Attention().to(device)
print(model)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        """
        Initialize Focal Loss.

        :param alpha: Class weights, a Tensor of shape (num_classes,).
                      For example, [1.5, 0.5, 1.5] reduces the weight of the neutral class (index 1).
        :param gamma: Focusing parameter, typically set to 2.0.
                      A larger value forces the model to focus more on hard-to-classify examples.
        :param reduction: 'mean' or 'sum', specifies how to aggregate the loss over the batch.
        """
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

        # If alpha is provided, convert it to a tensor
        if alpha is not None:
            if isinstance(alpha, list):
                self.alpha = torch.tensor(alpha)
            else:
                self.alpha = alpha
        else:
            self.alpha = None

    def forward(self, inputs, targets):
        """
        :param inputs: Model's predicted outputs (logits before softmax), shape (batch_size, num_classes).
        :param targets: Ground truth labels, shape (batch_size,).
        """
        # Calculate the standard Cross Entropy loss (without reduction)
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')

        # Convert cross entropy loss to probability (pt)
        pt = torch.exp(-ce_loss)

        # Compute the core Focal Loss formula: (1 - pt)^gamma * CE_loss
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        # If alpha is set, apply the corresponding weights based on the true labels
        if self.alpha is not None:
            self.alpha = self.alpha.to(inputs.device)
            # Gather the alpha weight corresponding to each sample's label
            alpha_t = self.alpha.gather(0, targets)
            focal_loss = alpha_t * focal_loss

        # Return the loss based on the specified reduction method
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:
# Assuming neutral news is abundant, while positive and negative news are minority classes
# Assign a weight of 1.5 to negative (index 0), 0.5 to neutral (index 1), and 1.5 to positive (index 2)
alpha_weights = [1.5, 0.5, 1.5]

# Instantiate the Focal Loss function
criterion_focal = FocalLoss(alpha=alpha_weights, gamma=2.0).to(device)

# The control group required by the project guidelines: standard Cross Entropy Loss
criterion_ce = nn.CrossEntropyLoss().to(device)

## Train the model

### Two Loss Function:Focal Loss and CE Loss model
Same learning rate and traing epochs for model training

In [ ]:
import torch
import os
from torch.optim import AdamW
from tqdm import tqdm
import torch.nn as nn

In [ ]:
def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer,
                       num_epochs=3, device='cuda', model_save_path='best_model.bin',
                       test_loader=None):

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [],   'val_acc': [],
        'test_acc': []
    }

    best_val_acc = 0.0

    for epoch in range(num_epochs):
        print(f"\n Epoch {epoch+1} / {num_epochs}")

        # Training
        model.train()
        total_train_loss = 0
        correct_train = 0
        total_train = 0

        progress_bar = tqdm(train_loader, desc=f"Training Epoch {epoch+1}")
        for batch in progress_bar:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            label          = batch['label'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, label)
            total_train_loss += loss.item()

            loss.backward()
            optimizer.step()

            predictions   = torch.argmax(logits, dim=1)
            correct_train += (predictions == label).sum().item()
            total_train   += label.size(0)
            progress_bar.set_postfix({'loss': loss.item()})

        avg_train_loss = total_train_loss / len(train_loader)
        avg_train_acc  = correct_train / total_train

        # Validation
        model.eval()
        total_val_loss = 0
        correct_val    = 0
        total_val      = 0

        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                label          = batch['label'].to(device)

                logits = model(input_ids, attention_mask)
                loss   = criterion(logits, label)
                total_val_loss += loss.item()

                predictions = torch.argmax(logits, dim=1)
                correct_val += (predictions == label).sum().item()
                total_val   += label.size(0)

        avg_val_loss = total_val_loss / len(val_loader)
        avg_val_acc  = correct_val / total_val

        # Test (per epoch)
        if test_loader is not None:
            correct_test = 0
            total_test   = 0
            with torch.no_grad():
                for batch in test_loader:
                    input_ids      = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    label          = batch['label'].to(device)

                    logits      = model(input_ids, attention_mask)
                    predictions = torch.argmax(logits, dim=1)
                    correct_test += (predictions == label).sum().item()
                    total_test   += label.size(0)
            avg_test_acc = correct_test / total_test
        else:
            avg_test_acc = 0.0

        # Record
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(avg_train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(avg_val_acc)
        history['test_acc'].append(avg_test_acc)

        print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.4f}")
        print(f"Val   Loss: {avg_val_loss:.4f} | Val   Acc: {avg_val_acc:.4f}")
        print(f"Test  Acc:  {avg_test_acc:.4f}")

        # Save best model
        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            print(f"  ✓ Best model updated (Val Acc={best_val_acc:.4f}), saving to {model_save_path}")
            torch.save(model.state_dict(), model_save_path)

    return model, history


# Hyperparameters
LEARNING_RATE = 2e-5
EPOCHS        = 3
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Experiment 1: Focal Loss
print("\n=== Experiment 1: Focal Loss ===")
model_focal     = FinBert_BiLSTM_Attention().to(device)
optimizer_focal = AdamW(model_focal.parameters(), lr=LEARNING_RATE)
criterion_focal = FocalLoss(alpha=[1.5, 0.5, 1.5], gamma=2.0).to(device)

_, history_focal = train_and_evaluate(
    model           = model_focal,
    train_loader    = train_dataloader,
    val_loader      = val_dataloader,
    test_loader     = test_dataloader,
    criterion       = criterion_focal,
    optimizer       = optimizer_focal,
    num_epochs      = EPOCHS,
    device          = device,
    model_save_path = 'best_model_focal.bin'
)

# Experiment 2: CrossEntropy Loss
print("\n=== Experiment 2: CrossEntropy Loss ===")
model_ce     = FinBert_BiLSTM_Attention().to(device)
optimizer_ce = AdamW(model_ce.parameters(), lr=LEARNING_RATE)
criterion_ce = nn.CrossEntropyLoss().to(device)

_, history_ce = train_and_evaluate(
    model           = model_ce,
    train_loader    = train_dataloader,
    val_loader      = val_dataloader,
    test_loader     = test_dataloader,
    criterion       = criterion_ce,
    optimizer       = optimizer_ce,
    num_epochs      = EPOCHS,
    device          = device,
    model_save_path = 'best_model_ce.bin'
)

print("\n Both experiments done. Best weights saved.")

### Generate 2 figures of different loss function


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    accuracy_score
)

# Figure 1: Focal Loss model - train loss, train acc, test acc vs epochs
# Figure 2: CE Loss model    - train loss, train acc, test acc vs epochs
# test_acc is recorded at each epoch during training (not a single scalar)

def plot_single_model_history(history, title, save_path=None):
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14)

    # Left plot: Training and validation loss over epochs
    axes[0].plot(epochs, history["train_loss"], marker='o', label="Train Loss")
    axes[0].plot(epochs, history["val_loss"],   marker='s', label="Val Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss vs Epochs")
    axes[0].legend()
    axes[0].grid(True)

    # Right plot: Train, validation and test accuracy over epochs
    axes[1].plot(epochs, history["train_acc"], marker='o', label="Train Acc")
    axes[1].plot(epochs, history["val_acc"],   marker='s', label="Val Acc")
    axes[1].plot(epochs, history["test_acc"],  marker='^', label="Test Acc")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Accuracy vs Epochs")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# Plot Figure 1: Focal Loss training curves
plot_single_model_history(history_focal,
                          "Figure 1: Focal Loss - Training Curves",
                          save_path="fig1_focal_loss_curves.png")

# Plot Figure 2: CrossEntropy Loss training curves
plot_single_model_history(history_ce,
                          "Figure 2: CrossEntropy Loss - Training Curves",
                          save_path="fig2_ce_loss_curves.png")

# Download figures
from google.colab import files
files.download("fig1_focal_loss_curves.png")
files.download("fig2_ce_loss_curves.png")

### Learning Rate Experiment
Training loss, training accuracy, and test accuracy with the change of number of epochs by using different scales of learning rate (e.g., 0.1, 0.01, 0.001, 0.0001) and other settings are the same as (1) and (2)

In [ ]:
# Learning rates tested: 0.1, 0.01, 0.001, 0.0001

lr_list = [0.1, 0.01, 0.001, 0.0001]
lr_histories_focal = {}
lr_histories_ce = {}

# Focal Loss: train with different learning rates
for lr in lr_list:
    print(f"\n Training with LR = {lr} (Focal Loss)")
    torch.cuda.empty_cache()

    model_lr     = FinBert_BiLSTM_Attention().to(device)
    optimizer_lr = AdamW(model_lr.parameters(), lr=lr)
    criterion_lr = FocalLoss(alpha=[1.5, 0.5, 1.5], gamma=2.0).to(device)

    _, history_lr = train_and_evaluate(
        model           = model_lr,
        train_loader    = train_dataloader,
        val_loader      = val_dataloader,
        test_loader     = test_dataloader,
        criterion       = criterion_lr,
        optimizer       = optimizer_lr,
        num_epochs      = 3,
        device          = device,
        model_save_path = f"best_model_focal_lr_{lr}.bin"
    )
    lr_histories_focal[lr] = history_lr

    # Release GPU memory after each run
    del model_lr
    torch.cuda.empty_cache()

# CrossEntropy Loss: train with different learning rates
for lr in lr_list:
    print(f"\n===== Training with LR = {lr} (CE Loss) =====")
    torch.cuda.empty_cache()

    model_lr     = FinBert_BiLSTM_Attention().to(device)
    optimizer_lr = AdamW(model_lr.parameters(), lr=lr)
    criterion_lr = nn.CrossEntropyLoss().to(device)

    _, history_lr = train_and_evaluate(
        model           = model_lr,
        train_loader    = train_dataloader,
        val_loader      = val_dataloader,
        test_loader     = test_dataloader,
        criterion       = criterion_lr,
        optimizer       = optimizer_lr,
        num_epochs      = 3,
        device          = device,
        model_save_path = f"best_model_ce_lr_{lr}.bin"
    )
    lr_histories_ce[lr] = history_lr

    # Release GPU memory after each run
    del model_lr
    torch.cuda.empty_cache()

# Figure 3: Focal Loss - different learning rates
epochs = range(1, 4)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Figure 3: Different Learning Rates (Focal Loss)", fontsize=14)

for lr in lr_list:
    axes[0].plot(epochs, lr_histories_focal[lr]["train_loss"], marker='o', label=f"LR={lr}")
    axes[1].plot(epochs, lr_histories_focal[lr]["train_acc"],  marker='o', label=f"LR={lr}")
    axes[2].plot(epochs, lr_histories_focal[lr]["test_acc"],   marker='^', label=f"LR={lr}")

for ax, title, ylabel in zip(axes,
    ["Train Loss", "Train Accuracy", "Test Accuracy"],
    ["Loss", "Accuracy", "Accuracy"]):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("fig3_focal_lr.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 4: CrossEntropy Loss - different learning rates
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Figure 4: Different Learning Rates (CrossEntropy Loss)", fontsize=14)

for lr in lr_list:
    axes[0].plot(epochs, lr_histories_ce[lr]["train_loss"], marker='o', label=f"LR={lr}")
    axes[1].plot(epochs, lr_histories_ce[lr]["train_acc"],  marker='o', label=f"LR={lr}")
    axes[2].plot(epochs, lr_histories_ce[lr]["test_acc"],   marker='^', label=f"LR={lr}")

for ax, title, ylabel in zip(axes,
    ["Train Loss", "Train Accuracy", "Test Accuracy"],
    ["Loss", "Accuracy", "Accuracy"]):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("fig4_ce_lr.png", dpi=150, bbox_inches='tight')
plt.show()

# Download figures
from google.colab import files
files.download("fig3_focal_lr.png")
files.download("fig4_ce_lr.png")

### Batch Size Experiment
Training loss, training accuracy, and test accuracy with the change of number of epochs by using different batch sizes (8, 16, 32, 64, ) and other settings are the same as (1) and
(2).

In [ ]:
# Fix loss function and learning rate, vary batch size to observe impact on training

bs_list = [8, 16, 32, 64]
bs_histories_focal = {}
bs_histories_ce = {}

# Focal Loss: train with different batch sizes
for bs in bs_list:
    print(f"\n Training with Batch Size = {bs} (Focal Loss)")
    torch.cuda.empty_cache()

    # Rebuild dataloaders with current batch size
    train_dl_bs = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=bs)
    val_dl_bs   = DataLoader(tokenized_datasets["validation"], batch_size=bs)
    test_dl_bs  = DataLoader(tokenized_datasets["test"], batch_size=bs)

    model_bs     = FinBert_BiLSTM_Attention().to(device)
    # Fixed LR same as Experiment 1
    optimizer_bs = AdamW(model_bs.parameters(), lr=2e-5)
    criterion_bs = FocalLoss(alpha=[1.5, 0.5, 1.5], gamma=2.0).to(device)

    _, history_bs = train_and_evaluate(
        model           = model_bs,
        train_loader    = train_dl_bs,
        val_loader      = val_dl_bs,
        test_loader     = test_dl_bs,
        criterion       = criterion_bs,
        optimizer       = optimizer_bs,
        num_epochs      = 3,
        device          = device,
        model_save_path = f"best_model_focal_bs_{bs}.bin"
    )
    bs_histories_focal[bs] = history_bs

    # Release GPU memory after each run
    del model_bs
    torch.cuda.empty_cache()

# CrossEntropy Loss: train with different batch sizes
for bs in bs_list:
    print(f"\n Training with Batch Size = {bs} (CE Loss)")
    torch.cuda.empty_cache()

    # Rebuild dataloaders with current batch size
    train_dl_bs = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=bs)
    val_dl_bs   = DataLoader(tokenized_datasets["validation"], batch_size=bs)
    test_dl_bs  = DataLoader(tokenized_datasets["test"], batch_size=bs)

    model_bs     = FinBert_BiLSTM_Attention().to(device)
    optimizer_bs = AdamW(model_bs.parameters(), lr=2e-5)  # Fixed LR same as Experiment 2
    criterion_bs = nn.CrossEntropyLoss().to(device)

    _, history_bs = train_and_evaluate(
        model           = model_bs,
        train_loader    = train_dl_bs,
        val_loader      = val_dl_bs,
        test_loader     = test_dl_bs,
        criterion       = criterion_bs,
        optimizer       = optimizer_bs,
        num_epochs      = 3,
        device          = device,
        model_save_path = f"best_model_ce_bs_{bs}.bin"
    )
    bs_histories_ce[bs] = history_bs

    # Release GPU memory after each run
    del model_bs
    torch.cuda.empty_cache()

# Figure 5: Focal Loss - different batch sizes
epochs = range(1, 4)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Figure 5: Different Batch Sizes (Focal Loss)", fontsize=14)

for bs in bs_list:
    axes[0].plot(epochs, bs_histories_focal[bs]["train_loss"], marker='o', label=f"BS={bs}")
    axes[1].plot(epochs, bs_histories_focal[bs]["train_acc"],  marker='o', label=f"BS={bs}")
    axes[2].plot(epochs, bs_histories_focal[bs]["test_acc"],   marker='^', label=f"BS={bs}")

for ax, title, ylabel in zip(axes,
    ["Train Loss", "Train Accuracy", "Test Accuracy"],
    ["Loss", "Accuracy", "Accuracy"]):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("fig5_focal_bs.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 6: CE Loss - different batch sizes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Figure 6: Different Batch Sizes (CrossEntropy Loss)", fontsize=14)

for bs in bs_list:
    axes[0].plot(epochs, bs_histories_ce[bs]["train_loss"], marker='o', label=f"BS={bs}")
    axes[1].plot(epochs, bs_histories_ce[bs]["train_acc"],  marker='o', label=f"BS={bs}")
    axes[2].plot(epochs, bs_histories_ce[bs]["test_acc"],   marker='^', label=f"BS={bs}")

for ax, title, ylabel in zip(axes,
    ["Train Loss", "Train Accuracy", "Test Accuracy"],
    ["Loss", "Accuracy", "Accuracy"]):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("fig6_ce_bs.png", dpi=150, bbox_inches='tight')
plt.show()

# Download figures
from google.colab import files
files.download("fig5_focal_bs.png")
files.download("fig6_ce_bs.png")

## First 100 results
Visualize your predicted labels together with their corresponding inputs and actual labels of the first 100 results in the test set

In [ ]:
# Model Evaluation on Test Set

# Define the evaluation function
def evaluate_model(model, model_path, test_loader, criterion, device):
    # Load the best saved model weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    all_preds       = []
    all_labels      = []
    total_test_loss = 0

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            total_test_loss += loss.item()

            # Get predicted class index
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_test_loss = total_test_loss / len(test_loader)
    test_acc      = accuracy_score(all_labels, all_preds)

    return all_labels, all_preds, avg_test_loss, test_acc


# Re-instantiate model structures for evaluation
model_focal_eval = FinBert_BiLSTM_Attention().to(device)
model_ce_eval    = FinBert_BiLSTM_Attention().to(device)

# Evaluate Focal Loss model using best saved weights
focal_labels, focal_preds, focal_test_loss, focal_test_acc = evaluate_model(
    model       = model_focal_eval,
    model_path  = "best_model_focal.bin",
    test_loader = test_dataloader,
    criterion   = criterion_focal,
    device      = device
)

# Evaluate CrossEntropy Loss model using best saved weights
ce_labels, ce_preds, ce_test_loss, ce_test_acc = evaluate_model(
    model       = model_ce_eval,
    model_path  = "best_model_ce.bin",
    test_loader = test_dataloader,
    criterion   = criterion_ce,
    device      = device
)

print("Focal Loss  - Test Loss: {:.4f} | Test Accuracy: {:.4f}".format(focal_test_loss, focal_test_acc))
print("CE Loss     - Test Loss: {:.4f} | Test Accuracy: {:.4f}".format(ce_test_loss, ce_test_acc))

In [ ]:

model_focal_eval = FinBert_BiLSTM_Attention().to(device)
model_ce_eval = FinBert_BiLSTM_Attention().to(device)

focal_labels, focal_preds, focal_test_loss, focal_test_acc = evaluate_model(
    model=model_focal_eval,
    model_path="best_model_focal.bin",
    test_loader=test_dataloader,
    criterion=criterion_focal,
    device=device
)

ce_labels, ce_preds, ce_test_loss, ce_test_acc = evaluate_model(
    model=model_ce_eval,
    model_path="best_model_ce.bin",
    test_loader=test_dataloader,
    criterion=criterion_ce,
    device=device
)

print("Focal Loss Test Accuracy:", round(focal_test_acc, 4))
print("CE Loss Test Accuracy:", round(ce_test_acc, 4))

In [ ]:

test_result_df = test_df.copy().reset_index(drop=True)

# Convert the real labels into numbers and then match them with the predictions
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

test_result_df["true_label"] = test_result_df["label"].str.lower().map(label2id)
test_result_df["pred_label_focal"] = [id2label[p] for p in focal_preds]
test_result_df["pred_label_ce"] = [id2label[p] for p in ce_preds]

output_100 = test_result_df[["text", "label", "pred_label_focal", "pred_label_ce"]].head(100)
output_100.columns = ["text", "true_label", "pred_label_focal", "pred_label_ce"]

print(output_100.head(10))

output_100.to_csv("test_predictions_top100.csv", index=False, encoding="utf-8-sig")
print("Saved: test_predictions_top100.csv")

In [ ]:
# ========== Figure 7: First 100 Test Predictions ==========

color_map = {
    "positive": "#c8e6c9",  # green
    "neutral":  "#fff9c4",  # yellow
    "negative": "#ffcdd2"   # red
}

fig, ax = plt.subplots(figsize=(16, 50))
ax.axis('off')

col_labels = ["#", "News Headline", "True Label", "Pred (Focal)", "Pred (CE)", "Focal correct?", "CE correct?"]
table_data = []

for i, row in output_100.iterrows():
    text_short    = row["text"][:80] + "..." if len(row["text"]) > 80 else row["text"]
    focal_correct = "Yes" if row["true_label"] == row["pred_label_focal"] else "No"
    ce_correct    = "Yes" if row["true_label"] == row["pred_label_ce"]    else "No"
    table_data.append([
        i + 1,
        text_short,
        row["true_label"],
        row["pred_label_focal"],
        row["pred_label_ce"],
        focal_correct,
        ce_correct
    ])

table = ax.table(
    cellText  = table_data,
    colLabels = col_labels,
    loc       = 'center',
    cellLoc   = 'left'
)

table.auto_set_font_size(False)
table.set_fontsize(7)
table.scale(1, 1.3)
table.auto_set_column_width([0, 1, 2, 3, 4, 5, 6])

# Header style
for j in range(7):
    table[0, j].set_facecolor("#4472C4")
    table[0, j].set_text_props(color='white', fontweight='bold')

# Color each row by true label, highlight incorrect predictions in red
for i, row in enumerate(table_data):
    bg_color = color_map.get(row[2], "white")
    for j in range(7):
        table[i + 1, j].set_facecolor(bg_color)
    if row[5] == "No":
        table[i + 1, 5].set_text_props(color='red', fontweight='bold')
    if row[6] == "No":
        table[i + 1, 6].set_text_props(color='red', fontweight='bold')

plt.title("Figure 7: First 100 Test Set Predictions (Focal Loss vs CE Loss)",
          fontsize=13, pad=20, fontweight='bold')
plt.savefig("fig7_top100_predictions.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig7_top100_predictions.png")

from google.colab import files
files.download("fig7_top100_predictions.png")

## Test by using FastAPI

In [ ]:
def predict(text):
    device = next(model.parameters()).device

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        if isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        pred = torch.argmax(logits, dim=1).item()

    labels = ["negative", "positive", "neutral"]
    return labels[pred]

In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok feedparser

In [ ]:
from fastapi import FastAPI
import feedparser

app = FastAPI()

@app.get("/")
def home():
    return {"message": "Financial Sentiment API is running"}

@app.get("/predict")
def predict_api(text: str):
    try:
        result = predict(text)
        return {
            "input": text,
            "prediction": result
        }
    except Exception as e:
        return {"error": str(e)}

@app.get("/latest_news_sentiment")
def latest_news_sentiment():
    try:
        rss_url = "https://news.google.com/rss/search?q=financial+news&hl=en-US&gl=US&ceid=US:en"
        feed = feedparser.parse(rss_url)

        results = []
        for entry in feed.entries[:5]:
            title = entry.title
            sentiment = predict(title)
            results.append({
                "headline": title,
                "prediction": sentiment
            })

        return results
    except Exception as e:
        return {"error": str(e)}

In [ ]:
import threading
import uvicorn

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run)
thread.start()

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3AUhHYGppSb1Q2kkA63ZaiLSAjS_65bVxW87EsjMyfmQAtDp8")
public_url = ngrok.connect(8000)
print(public_url)

In [ ]:
import requests

# Replace with your ngrok public_url
BASE_URL = "https://nonspinal-resistingly-harold.ngrok-free.dev"

#  Test 1: Single text prediction
test_sentences = [
    "Apple reported record profits this quarter",
    "The company filed for bankruptcy after massive losses",
    "The stock market remained stable today",
    "Goldman Sachs upgrades Tesla to buy rating",
    "Oil prices plunge amid global recession fears",
    "Federal Reserve holds interest rates steady"
]

print(f"{'News Headline':<48} {'Sentiment':>12}")


for text in test_sentences:
    response = requests.get(f"{BASE_URL}/predict", params={"text": text})
    result   = response.json()["prediction"]
    print(f"{text[:48]:<48} {result:>12}")


# Test 2: Fetch latest financial news and predict
print("\nTest2: Latest Financial News Sentiment")
response     = requests.get(f"{BASE_URL}/latest_news_sentiment")
news_results = response.json()


print(f"{'Sentiment':<12} {'Headline'}")
for item in news_results:
    headline  = item["headline"][:80]
    sentiment = item["prediction"]
    print(f"{sentiment:<12} {headline}")
